## 📦 Section 1: Install Dependencies

In [ ]:
!pip install -q langchain langgraph langchain-groq tavily-python requests
!pip install -q markdown weasyprint
!apt-get install -y weasyprint 2>/dev/null | tail -1
print('✅ All packages installed')

0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
✅ All packages installed


In [ ]:
import os, json, re
import sys

# Attempt Colab Secret loading safely
try:
    from google.colab import userdata
    GROQ_API_KEY      = userdata.get('GROQ_API_KEY')
    TAVILY_API_KEY    = userdata.get('TAVILY_API_KEY')
    META_ACCESS_TOKEN = userdata.get('META_ACCESS_TOKEN')
    SEARCHAPI_KEY     = userdata.get('SEARCHAPI_KEY')
    print('✅ API keys loaded from Colab Secrets')
except ImportError:
    # Check if a local .env file exists and parse it
    if os.path.exists('.env'):
        with open('.env', 'r', encoding='utf-8') as f:
            for line in f:
                if '=' in line and not line.startswith('#'):
                    k, v = line.strip().split('=', 1)
                    os.environ[k.strip()] = v.strip().strip('"').strip("'")
                    
    GROQ_API_KEY      = os.getenv('GROQ_API_KEY', '')
    TAVILY_API_KEY    = os.getenv('TAVILY_API_KEY', '')
    META_ACCESS_TOKEN = os.getenv('META_ACCESS_TOKEN', '')
    SEARCHAPI_KEY     = os.getenv('SEARCHAPI_KEY', '')
    print('ℹ️  Running locally - loaded keys from env/local .env file')

# Warn if keys are missing
missing_keys = [k for k, v in {
    'GROQ_API_KEY': GROQ_API_KEY,
    'TAVILY_API_KEY': TAVILY_API_KEY,
    'SEARCHAPI_KEY': SEARCHAPI_KEY
}.items() if not v]

if missing_keys:
    print(f'⚠️ Warning: The following keys are missing: {", ".join(missing_keys)}')

os.environ['GROQ_API_KEY']   = GROQ_API_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

CONFIG = {
    'groq_model'         : 'llama-3.3-70b-versatile',
    'temperature'        : 0.2,
    'max_ads_to_fetch'   : 20,
    'max_search_results' : 5,
    'ad_type'             : 'ALL',
    'meta_url'           : 'https://graph.facebook.com/v20.0/ads_archive',
    'cta_keywords'       : [
        'shop now', 'buy now', 'order now', 'daftar', 'coba gratis',
        'pelajari', 'hubungi', 'download', 'daftar sekarang', 'beli sekarang',
        'dapatkan', 'klaim', 'mulai', 'cek', 'lihat'
    ]
}

print(f'\n🤖 Model  : {CONFIG["groq_model"]}')
print(f'📊 Max ads: {CONFIG["max_ads_to_fetch"]}')

⚠️  Using placeholder keys — add real keys via Colab Secrets

🤖 Model  : llama-3.3-70b-versatile
📊 Max ads: 20


## 📊 Section 3: Metrics & Analysis Functions
Rule-based analysis functions that extract patterns from raw ad data.


In [ ]:
from collections import Counter
from typing import List, Dict


def extract_ad_texts(ads: List[Dict]) -> List[str]:
    """Extract all text content from ad records."""
    texts = []
    for ad in ads:
        parts = (
            ad.get('ad_creative_bodies', []) +
            ad.get('ad_creative_link_titles', []) +
            ad.get('ad_creative_link_descriptions', [])
        )
        text = ' '.join(parts).strip()
        if text:
            texts.append(text)
    return texts


def detect_cta(texts: List[str]) -> Dict:
    """Detect CTA patterns in ad copy."""
    cta_counts = Counter()
    ads_with_cta = 0
    for text in texts:
        text_lower = text.lower()
        has_cta = False
        for cta in CONFIG['cta_keywords']:
            if cta in text_lower:
                cta_counts[cta] += 1
                has_cta = True
        if has_cta:
            ads_with_cta += 1
            
    total = len(texts) or 1
    return {
        'cta_frequency'  : dict(cta_counts.most_common(10)),
        'top_cta'        : cta_counts.most_common(1)[0][0] if cta_counts else 'none detected',
        'ads_with_cta_pct': round(ads_with_cta / total * 100, 1)
    }


def analyze_copy_length(texts: List[str]) -> Dict:
    """Analyze ad copy length patterns."""
    if not texts:
        return {}
    lengths = [len(t.split()) for t in texts]
    short   = sum(1 for l in lengths if l <= 20)
    medium  = sum(1 for l in lengths if 21 <= l <= 60)
    long_   = sum(1 for l in lengths if l > 60)
    total   = len(lengths)
    return {
        'avg_word_count'     : round(sum(lengths) / total, 1),
        'min_word_count'     : min(lengths),
        'max_word_count'     : max(lengths),
        'short_copy_pct'     : round(short  / total * 100, 1),
        'medium_copy_pct'    : round(medium / total * 100, 1),
        'long_copy_pct'      : round(long_  / total * 100, 1),
        'dominant_format'    : 'short' if short >= medium and short >= long_ else
                               'medium' if medium >= long_ else 'long'
    }


def detect_promo_signals(texts: List[str]) -> Dict:
    """Detect promotional signals like discounts, free shipping, etc."""
    promo_keywords = [
        'diskon', 'gratis', 'free', 'promo', 'cashback', 'voucher',
        'hemat', 'murah', 'terjangkau', 'flash sale', 'limited',
        '%', 'off', 'ongkir', 'freeship'
    ]
    promo_counts = Counter()
    ads_with_promo = 0
    for text in texts:
        text_lower = text.lower()
        found = False
        for kw in promo_keywords:
            if kw in text_lower:
                promo_counts[kw] += 1
                found = True
        if found:
            ads_with_promo += 1
    total = len(texts) or 1
    return {
        'ads_with_promo_pct' : round(ads_with_promo / total * 100, 1),
        'top_promo_keywords' : dict(promo_counts.most_common(5)),
    }


def compute_metrics(ads: List[Dict]) -> Dict:
    """Aggregate all metrics from raw ad data."""
    texts       = extract_ad_texts(ads)
    page_names  = [ad.get('page_name', 'Unknown') for ad in ads]
    unique_pages = list(set(page_names))

    cta_analysis   = detect_cta(texts)
    copy_analysis  = analyze_copy_length(texts)
    promo_analysis = detect_promo_signals(texts)

    return {
        'total_ads_analyzed' : len(ads),
        'unique_advertisers' : len(unique_pages),
        'advertiser_names'   : unique_pages[:10],
        'cta_analysis'       : cta_analysis,
        'copy_length'        : copy_analysis,
        'promo_signals'      : promo_analysis,
        'sample_ad_texts'    : texts[:5],
    }


print('✅ Metrics functions ready')

✅ Metrics functions ready


## 🛠️ Section 4: Agent Tool Definitions


In [ ]:
import requests
from langchain_core.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

# Store raw ads globally so metrics can be computed after agent run
_raw_ads_cache = []


@tool
def fetch_meta_ads(search_query: str, country: str = 'ID') -> str:
    """
    Fetch active ads from Meta Ad Library using SearchAPI.io.
    More reliable than direct Meta Graph API for keyword searches.

    Args:
        search_query: Brand name or keyword (e.g. 'Tokopedia', 'cekat ai')
        country: Country code (default: ID for Indonesia)
    """
    global _raw_ads_cache
    try:
        params = {
            'engine'       : 'meta_ad_library',
            'q'            : search_query,
            'country'      : country,
            'ad_type'      : 'all',
            'active_status': 'active',
            'api_key'      : SEARCHAPI_KEY
        }
        response = requests.get('https://www.searchapi.io/api/v1/search', params=params)
        data     = response.json()

        if 'error' in data:
            return f"SearchAPI Error: {data['error']}"

        ads = data.get('ads', [])
        if not ads:
            return json.dumps({'error': f"No active ads found for '{search_query}'"})

        # Cache for metrics
        _raw_ads_cache = ads

        # Build clean records for LLM
        records = []
        for ad in ads:
            snap = ad.get('snapshot', {})
            body = snap.get('body', {}).get('text', '')
            records.append({
                'page_name'  : ad.get('page_name', 'Unknown'),
                'is_active'  : ad.get('is_active', False),
                'start_date' : ad.get('start_date', '')[:10],
                'platforms'  : ad.get('publisher_platform', []),
                'body'       : body[:300],
                'title'      : snap.get('title', ''),
                'cta_text'   : snap.get('cta_text', ''),
                'cta_type'   : snap.get('cta_type', ''),
                'media_type' : snap.get('display_format', ''),
            })

        return json.dumps({
            'query'        : search_query,
            'country'      : country,
            'total_results': data.get('search_information', {}).get('total_results', len(records)),
            'total_fetched': len(records),
            'ads'          : records
        }, ensure_ascii=False, indent=2)

    except Exception as e:
        return f"Error: {str(e)}"


@tool
def fetch_ads_by_page_id(page_id: str, country: str = 'ID') -> str:
    """
    Fetch active ads from a specific page using SearchAPI.io.
    Works with both Facebook Pages and personal/business profiles.

    Args:
        page_id: Facebook Page ID or profile ID
        country : Country code (default: ID)
    """
    global _raw_ads_cache
    try:
        params = {
            'engine'       : 'meta_ad_library',
            'page_id'      : page_id,
            'country'      : country,
            'ad_type'      : 'all',
            'active_status': 'active',
            'api_key'      : SEARCHAPI_KEY
        }
        response = requests.get('https://www.searchapi.io/api/v1/search', params=params)
        data     = response.json()

        if 'error' in data:
            return f"SearchAPI Error: {data['error']}"

        ads = data.get('ads', [])
        if not ads:
            return json.dumps({'error': f"No active ads found for page_id '{page_id}'"})

        _raw_ads_cache = ads

        records = []
        for ad in ads:
            snap = ad.get('snapshot', {})
            body = snap.get('body', {}).get('text', '')
            records.append({
                'page_name'  : ad.get('page_name', 'Unknown'),
                'is_active'  : ad.get('is_active', False),
                'start_date' : ad.get('start_date', '')[:10],
                'platforms'  : ad.get('publisher_platform', []),
                'body'       : body[:300],
                'title'      : snap.get('title', ''),
                'cta_text'   : snap.get('cta_text', ''),
                'cta_type'   : snap.get('cta_type', ''),
                'media_type' : snap.get('display_format', ''),
            })

        return json.dumps({
            'page_id'      : page_id,
            'country'      : country,
            'total_results': data.get('search_information', {}).get('total_results', len(records)),
            'total_fetched': len(records),
            'ads'          : records
        }, ensure_ascii=False, indent=2)

    except Exception as e:
        return f"Error: {str(e)}"


@tool
def web_search(query: str) -> str:
    """
    Search the web for additional context about a brand or competitor.
    Use to enrich ad analysis with brand background and marketing strategy.

    Args:
        query: Search query (e.g. 'Tokopedia marketing strategy 2025')
    """
    try:
        results   = tavily_client.search(
            query=query,
            max_results=CONFIG['max_search_results'],
            search_depth='basic'
        )
        formatted = [{
            'title'  : r.get('title', ''),
            'url'    : r.get('url', ''),
            'content': r.get('content', '')[:400]
        } for r in results.get('results', [])]
        return json.dumps({'query': query, 'results': formatted}, ensure_ascii=False, indent=2)
    except Exception as e:
        return f"Error: {str(e)}"


tools = [fetch_meta_ads, fetch_ads_by_page_id, web_search]
print(f'✅ Tools updated with SearchAPI.io: {[t.name for t in tools]}')


✅ Tools updated with SearchAPI.io: ['fetch_meta_ads', 'fetch_ads_by_page_id', 'web_search']


In [ ]:
# ── BONUS: Search by Page ID ───────────────────────────────────────

@tool
def fetch_ads_by_page_id(page_id: str, country: str = 'ID') -> str:
    """
    Fetch active ads from a specific Facebook Page using its Page ID.
    More accurate than keyword search — targets exact competitor page.

    Args:
        page_id: Facebook Page ID (e.g. '123456789')
        country : Country code (default: ID)
    """
    global _raw_ads_cache
    try:
        params = {
            'access_token'        : META_ACCESS_TOKEN,
            'search_page_ids'     : page_id,        # ← key difference
            'ad_reached_countries': [country],
            'ad_active_status'    : 'ACTIVE',
            'ad_type'             : 'ALL',
            'limit'               : CONFIG['max_ads_to_fetch'],
            'fields'              : 'id,ad_creation_time,page_name,'
                                    'ad_creative_bodies,ad_creative_link_titles,'
                                    'ad_creative_link_descriptions,ad_creative_link_captions'
        }
        response = requests.get(CONFIG['meta_url'], params=params)
        data     = response.json()

        if 'error' in data:
            return f"Meta API Error: {data['error']['message']}"

        ads = data.get('data', [])
        if not ads:
            return json.dumps({'error': f"No active ads found for page_id '{page_id}'"})

        _raw_ads_cache = ads

        records = []
        for ad in ads:
            records.append({
                'page_name'  : ad.get('page_name', 'Unknown'),
                'created'    : ad.get('ad_creation_time', '')[:10],
                'body'       : (ad.get('ad_creative_bodies') or [''])[:1][0][:300],
                'title'      : (ad.get('ad_creative_link_titles') or [''])[:1][0],
                'description': (ad.get('ad_creative_link_descriptions') or [''])[:1][0],
            })

        return json.dumps({
            'page_id'    : page_id,
            'country'    : country,
            'total_found': len(records),
            'ads'        : records
        }, ensure_ascii=False, indent=2)

    except Exception as e:
        return f"Error: {str(e)}"


# Update tools list
tools = [fetch_meta_ads, fetch_ads_by_page_id, web_search]
print(f'✅ Tools updated: {[t.name for t in tools]}')

✅ Tools updated: ['fetch_meta_ads', 'fetch_ads_by_page_id', 'web_search']


## 🤖 Section 5: LangGraph Agent


In [ ]:
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

SYSTEM_PROMPT = """
You are AdSpy, a senior paid media strategist with 10+ years experience running
Meta ad campaigns across Southeast Asia. You think like a performance marketer —
every insight must connect to a specific action that improves ROAS or reduces CPL.

WORKFLOW — always follow this order:
1. Use `fetch_meta_ads` to get active competitor ads
2. Use `web_search` to get brand background, positioning, and recent marketing news
3. Analyze all data thoroughly using the framework below
4. Produce the full strategic report

DEEP ANALYSIS FRAMEWORK:

A. CREATIVE ANALYSIS
   - What formats dominate? (single image, video, carousel, UGC-style)
   - What visual hooks are being used? (before/after, testimonial, product demo, lifestyle)
   - What emotions are being triggered? (fear, greed, aspiration, FOMO, curiosity)

B. COPY ANALYSIS
   - What is the dominant messaging angle?
     (promo/discount, pain point, aspiration, social proof, authority, urgency)
   - How is the headline structured? (question, statement, number, command)
   - What specific pain points or desires are being addressed?
   - What power words are frequently used?
   - Copy length pattern: short punchy vs long-form storytelling

C. OFFER ANALYSIS
   - What offers are being promoted? (discount %, free trial, freeship, cashback, bundle)
   - What is the primary CTA? (Shop Now, Learn More, Send Message, Sign Up)
   - Is there a clear value proposition or USP?

D. AUDIENCE SIGNALS
   - What platforms are ads running on? (FB only, IG only, cross-platform)
   - Any signs of retargeting vs cold traffic copy?
   - Language used: formal or casual, Bahasa Indonesia or English?

E. COMPETITIVE GAP ANALYSIS
   - What messaging angles are competitors NOT using?
   - What audience segments appear underserved?
   - What offers or formats are missing from the competitive landscape?

OUTPUT — always produce this exact structure:

## ADSPY INTELLIGENCE REPORT
**Target:** [brand] | **Country:** [country] | **Date:** [date]

### 1. Executive Summary
2-3 sentences capturing the most important competitive insight.

### 2. Ad Audit Overview
- Total active ads, platforms used, estimated activity level
- Key advertisers found (if multiple pages)

### 3. Creative & Messaging Analysis
Break down by angle with REAL examples quoted from actual ad copy.
For each dominant angle explain WHY it works psychologically.

### 4. Offer & CTA Analysis
What offers and CTAs dominate, with specific examples.
What this tells us about their funnel strategy.

### 5. Audience & Platform Strategy
Where they are running, language signals, cold vs warm traffic indicators.

### 6. Competitive Gap Analysis
Specific gaps with explanation of why each is an opportunity.
Prioritize gaps by potential impact (High / Medium / Low).

### 7. Actionable Recommendations
Exactly 5 recommendations. Each must follow this format:

**Recommendation [N]: [Title]**
- **Why:** [What competitor data supports this]
- **What to do:** [Specific, concrete action]
- **Expected impact:** [What metric this should improve and by roughly how much]
- **Priority:** High / Medium / Low

### 8. Quick Win (Do This First)
Single most impactful thing to do in the next 7 days based on the gaps found.

Be ruthlessly specific. Vague insights are useless to a media buyer.
Always reference actual ad copy examples when making claims.
"""

llm = ChatGroq(
    model=CONFIG['groq_model'],
    temperature=CONFIG['temperature'],
    api_key=GROQ_API_KEY
)

agent = create_react_agent(model=llm, tools=tools, prompt=SYSTEM_PROMPT)
print('✅ Agent re-initialized with 3 tools')
print(f'   Model : {CONFIG["groq_model"]}')
print(f'   Tools : {[t.name for t in tools]}')

✅ Agent re-initialized with 3 tools
   Model : llama-3.3-70b-versatile
   Tools : ['fetch_meta_ads', 'fetch_ads_by_page_id', 'web_search']


/tmp/ipykernel_61159/1796332874.py:95: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=tools, prompt=SYSTEM_PROMPT)


## 🚀 Section 6: Run the Agent


In [ ]:
def run_adspy(query: str, country: str = 'ID', verbose: bool = True):
    """
    Run the AdSpy Intelligence Agent.

    Args:
        query  : Brand name, keyword, or industry
        country: Country code (ID, SG, MY, US, etc.)
        verbose: Show agent reasoning steps
    """
    print(f'\n{"="*60}')
    print(f'🔍 AdSpy Analysis: "{query}" | Country: {country}')
    print(f'{"="*60}')

    user_message = f"""
    Analyze competitor ads for: "{query}"
    Target country: {country}

    Please:
    1. Fetch active ads from Meta Ad Library
    2. Search web for brand/competitor context
    3. Generate the full intelligence report
    """

    inputs       = {'messages': [HumanMessage(content=user_message)]}
    final_report = ''
    step_count   = 0

    for chunk in agent.stream(inputs, stream_mode='values'):
        last_msg = chunk['messages'][-1]
        if verbose and hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                step_count += 1
                print(f'⚙️  Step {step_count}: {tc["name"]}({list(tc["args"].keys())})')
        if hasattr(last_msg, 'content') and last_msg.content:
            if not hasattr(last_msg, 'tool_calls') or not last_msg.tool_calls:
                final_report = last_msg.content

    print(f'{"="*60}')
    print(final_report)
    return final_report


print('✅ run_adspy() ready')
print()
print('Usage examples:')
print('  run_adspy("Tokopedia")')
print('  run_adspy("skincare", country="ID")')
print('  run_adspy("online course", country="SG")')

✅ run_adspy() ready

Usage examples:
  run_adspy("Tokopedia")
  run_adspy("skincare", country="ID")
  run_adspy("online course", country="SG")


## 🎯 Section 7: Live Demo
> Change `TARGET` to any brand or keyword you want to analyze.


In [ ]:
# ── Section 7a: Helper — Cari Page ID dari nama brand ─────────────
# Jalankan cell ini dulu jika belum tau Page ID kompetitor

print('=' * 60)
print('Cari Page ID Kompetitor')
print('=' * 60)
print()

keyword = input('Masukkan nama brand kompetitor: ').strip()

params = {
    'engine' : 'meta_ad_library_page_search',
    'q'      : keyword,
    'country': 'ID',
    'ad_type': 'all',
    'api_key': SEARCHAPI_KEY
}

response = requests.get('https://www.searchapi.io/api/v1/search', params=params)
data     = response.json()
pages    = data.get('page_results', [])

if not pages:
    print('Tidak ada hasil — coba keyword lain')
else:
    print()
    print(f'{"No":<5} {"Nama Page":<40} {"Page ID":<20} {"Kategori"}')
    print('-' * 85)
    for i, page in enumerate(pages[:10]):
        name     = page.get('name', 'Unknown')[:38]
        page_id  = page.get('page_id', '-')
        category = page.get('category', '-')
        print(f'{i+1:<5} {name:<40} {page_id:<20} {category}')
    print()

    choice = input('Pilih nomor yang tepat (1-10): ').strip()
    try:
        selected = pages[int(choice) - 1]
        PAGE_ID  = selected.get('page_id', '')
        TARGET   = selected.get('name', '')
        print()
        print('Dipilih : ' + TARGET)
        print('Page ID : ' + PAGE_ID)
        print()
        print('Lanjut ke Section 7b, pilih mode 2, masukkan Page ID: ' + PAGE_ID)
    except (ValueError, IndexError):
        print('Nomor tidak valid')

In [ ]:
print('=' * 60)
print('AdSpy Intelligence Agent')
print('=' * 60)
print()
print('Pilih mode pencarian:')
print('  1 -> Search by Keyword (contoh: Shopee, skincare, online course)')
print('  2 -> Search by Page ID (lebih akurat, gunakan hasil Section 7a)')
print()

mode    = input('Masukkan pilihan (1/2): ').strip()
COUNTRY = input('Masukkan kode negara (default: ID): ').strip() or 'ID'

if mode == '2':
    PAGE_ID = input('Masukkan Page ID: ').strip()
    TARGET  = PAGE_ID
    print()
    print('Mode: Page ID -> ' + PAGE_ID + ' | Country: ' + COUNTRY)
    print()

    # Direct invoke — bypass agent, langsung pakai fetch_ads_by_page_id
    print('Fetching ads...')
    raw_result = fetch_ads_by_page_id.invoke({
        'page_id': PAGE_ID,
        'country': COUNTRY
    })

    # Pass hasil fetch langsung ke LLM untuk analisis
    print('Analyzing with LLM...')
    analysis_prompt = (
        'Analyze the following competitor ad data and generate a full intelligence report.\n'
        'Country: ' + COUNTRY + '\n\n'
        'Ad Data:\n' + raw_result + '\n\n'
        'Generate the full report following the exact output structure in your instructions.'
    )
    response = llm.invoke([HumanMessage(content=analysis_prompt)])
    report   = response.content
    print()
    print('=' * 60)
    print(report)

else:
    TARGET = input('Masukkan keyword atau nama brand: ').strip()
    print()
    print('Mode: Keyword -> ' + TARGET + ' | Country: ' + COUNTRY)
    report = run_adspy(TARGET, country=COUNTRY)

AdSpy Intelligence Agent

Pilih mode pencarian:
  1 -> Search by Keyword (contoh: Shopee, skincare, online course)
  2 -> Search by Page ID (lebih akurat, gunakan hasil Section 7a)

Masukkan pilihan (1/2): 2
Masukkan kode negara (default: ID): 
Masukkan Page ID: 131747220174107

Mode: Page ID -> 131747220174107 | Country: ID

Fetching ads...
Analyzing with LLM...

**Competitor Ad Intelligence Report**

**Country:** ID
**Page ID:** 131747220174107

**Ad Overview**
---------------

* **Active Ads:** 0
* **Error Message:** No active ads found for page_id '131747220174107'

**Ad Insights**
--------------

* **No ad data available** due to the absence of active ads for the specified page ID.

**Targeting and Budget**
----------------------

* **No data available** as there are no active ads to analyze.

**Ad Creative**
--------------

* **No ad creatives available** for review, as no active ads were found.

**Conclusion**
----------

Based on the provided ad data, it appears that the compet

## 📊 Section 8: Compute & Visualize Metrics
Rule-based metrics extracted from the fetched ad data.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'


# ── Adapter: convert SearchAPI format ke format metrics ────────────
def normalize_ads_for_metrics(ads):
    """Convert SearchAPI.io or direct Meta Graph API ad formats to metrics format."""
    normalized = []
    for ad in ads:
        if not ad:
            continue
            
        # 1. SearchAPI.io format (contains snapshot)
        if 'snapshot' in ad:
            snap  = ad.get('snapshot', {}) or {}
            body  = snap.get('body', {}).get('text', '') or ''
            title = snap.get('title', '') or ''
            
            card_texts = []
            for card in snap.get('cards', []):
                if card.get('body'):
                    card_texts.append(card['body'])
                if card.get('title'):
                    card_texts.append(card['title'])
                    
        # 2. Direct Meta Graph API format
        else:
            bodies = ad.get('ad_creative_bodies', []) or []
            body   = bodies[0] if bodies else ''
            
            titles = ad.get('ad_creative_link_titles', []) or []
            title  = titles[0] if titles else ''
            
            card_texts = ad.get('ad_creative_link_descriptions', []) or []

        normalized.append({
            'page_name'                    : ad.get('page_name', 'Unknown'),
            'ad_creative_bodies'           : [body] if body else [],
            'ad_creative_link_titles'      : [title] if title else [],
            'ad_creative_link_descriptions': [t for t in card_texts if t],
        })
    return normalized


# ── Compute metrics ────────────────────────────────────────────────
if _raw_ads_cache:
    normalized = normalize_ads_for_metrics(_raw_ads_cache)
    metrics    = compute_metrics(normalized)
    print('✅ Metrics computed!')
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print('⚠️  No ad data found — run Section 7 first')
    metrics = {}

⚠️  No ad data found — run Section 7 first


In [ ]:
def plot_metrics(metrics: dict, brand: str):
    """Visualize ad metrics as charts."""
    if not metrics:
        print('No metrics to plot')
        return

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        'AdSpy Metrics - ' + brand + '\n' +
        'Total Ads: ' + str(metrics.get('total_ads_analyzed', 0)) +
        ' | Advertisers: ' + str(metrics.get('unique_advertisers', 0)),
        fontsize=13, fontweight='bold'
    )

    # Chart 1: Copy Length Distribution
    cl = metrics.get('copy_length', {})
    if cl:
        labels = ['Short\n(<=20 words)', 'Medium\n(21-60)', 'Long\n(>60)']
        values = [
            cl.get('short_copy_pct', 0),
            cl.get('medium_copy_pct', 0),
            cl.get('long_copy_pct', 0)
        ]
        colors = ['#4361EE', '#7B8CDE', '#C5CCF5']
        bars = axes[0].bar(labels, values, color=colors, edgecolor='white', width=0.5)
        axes[0].set_title('Copy Length Distribution', fontsize=11)
        axes[0].set_ylabel('Percentage (%)')
        max_val = max(values) if max(values) > 0 else 100
        axes[0].set_ylim(0, max_val * 1.3)
        for bar, val in zip(bars, values):
            if val > 0:
                axes[0].text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 1,
                    str(val) + '%',
                    ha='center', va='bottom', fontsize=10
                )
    else:
        axes[0].text(0.5, 0.5, 'No copy data', ha='center', va='center',
                     transform=axes[0].transAxes)
        axes[0].set_title('Copy Length Distribution', fontsize=11)

    # Chart 2: Top CTAs
    cta = metrics.get('cta_analysis', {}).get('cta_frequency', {})
    if cta:
        top_ctas = dict(list(cta.items())[:6])
        axes[1].barh(
            list(top_ctas.keys()),
            list(top_ctas.values()),
            color='#4361EE', edgecolor='white'
        )
        axes[1].set_title('Top CTA Keywords', fontsize=11)
        axes[1].set_xlabel('Frequency')
        # Add value labels
        for i, val in enumerate(top_ctas.values()):
            axes[1].text(val + 0.1, i, str(val), va='center', fontsize=9)
    else:
        axes[1].text(0.5, 0.5, 'No CTA data', ha='center', va='center',
                     transform=axes[1].transAxes)
        axes[1].set_title('Top CTA Keywords', fontsize=11)

    # Chart 3: Promo Signals
    promo     = metrics.get('promo_signals', {})
    promo_pct = promo.get('ads_with_promo_pct', 0)
    no_promo  = round(100 - promo_pct, 1)

    if promo_pct > 0 or no_promo > 0:
        axes[2].pie(
            [promo_pct, no_promo],
            labels=['Has Promo\n' + str(promo_pct) + '%',
                    'No Promo\n' + str(no_promo) + '%'],
            colors=['#4361EE', '#E0E4FB'],
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2}
        )
    else:
        axes[2].text(0.5, 0.5, 'No promo data', ha='center', va='center',
                     transform=axes[2].transAxes)
    axes[2].set_title('Promo Signal Distribution', fontsize=11)

    plt.tight_layout()
    chart_file = 'adspy_chart_' + brand.replace(' ', '_') + '.png'
    plt.savefig(chart_file, dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved: ' + chart_file)
    return chart_file


if metrics:
    chart_file = plot_metrics(metrics, TARGET)
else:
    print('No metrics available — run Section 7 and Section 8 first')

No metrics available — run Section 7 and Section 8 first


## 📄 Section 9: Export PDF Report


In [ ]:
import markdown as md_lib
from weasyprint import HTML
from datetime import datetime


def export_pdf(report: str, brand: str, metrics: dict = None):
    """
    Convert markdown report to styled PDF and download/save.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    
    # Sanitize brand name for safe filenames
    safe_brand = re.sub(r'[^a-zA-Z0-9_-]', '_', brand)
    pdf_file   = f'adspy_report_{safe_brand}_{timestamp}.pdf'

    body_html = md_lib.markdown(report, extensions=['tables', 'fenced_code'])

    # Metrics summary block
    metrics_html = ''
    if metrics:
        cl    = metrics.get('copy_length', {})
        cta   = metrics.get('cta_analysis', {})
        promo = metrics.get('promo_signals', {})
        metrics_html = f"""
        <div class="metrics-box">
            <h3>📊 Metrics Summary</h3>
            <table>
                <tr><th>Metric</th><th>Value</th></tr>
                <tr><td>Total Ads Analyzed</td><td><b>{metrics.get('total_ads_analyzed', '-')}</b></td></tr>
                <tr><td>Unique Advertisers</td><td><b>{metrics.get('unique_advertisers', '-')}</b></td></tr>
                <tr><td>Avg Copy Length</td><td><b>{cl.get('avg_word_count', '-')} words</b></td></tr>
                <tr><td>Dominant Copy Format</td><td><b>{cl.get('dominant_format', '-')}</b></td></tr>
                <tr><td>Top CTA</td><td><b>{cta.get('top_cta', '-')}</b></td></tr>
                <tr><td>Ads with CTA</td><td><b>{cta.get('ads_with_cta_pct', '-')}%</b></td></tr>
                <tr><td>Ads with Promo Signal</td><td><b>{promo.get('ads_with_promo_pct', '-')}%</b></td></tr>
            </table>
        </div>
        """

    full_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body {{
                font-family: Arial, sans-serif;
                font-size: 13px;
                color: #1a1a2e;
                margin: 0; padding: 0;
            }}
            .header {{
                background: #4361EE;
                color: white;
                padding: 32px 40px 24px;
            }}
            .header h1 {{ margin: 0 0 4px 0; font-size: 22px; }}
            .header .sub {{ font-size: 12px; opacity: 0.85; margin-top: 6px; }}
            .content {{ padding: 32px 40px; }}
            h2 {{
                font-size: 15px; color: #4361EE;
                border-bottom: 2px solid #4361EE;
                padding-bottom: 5px; margin-top: 28px;
            }}
            h3 {{ font-size: 13px; color: #1a1a2e; margin-top: 18px; }}
            p, li {{ line-height: 1.7; color: #333; }}
            table {{ width: 100%; border-collapse: collapse; margin: 14px 0; font-size: 12px; }}
            th {{ background: #4361EE; color: white; padding: 8px 12px; text-align: left; }}
            td {{ padding: 7px 12px; border-bottom: 1px solid #e8e8e8; }}
            tr:nth-child(even) td {{ background: #f8f9ff; }}
            .metrics-box {{
                background: #f0f4ff;
                border-left: 4px solid #4361EE;
                padding: 16px 20px;
                margin: 20px 0;
                border-radius: 0 8px 8px 0;
            }}
            .metrics-box h3 {{ margin-top: 0; color: #4361EE; }}
            .footer {{
                margin-top: 40px; padding: 16px 40px;
                border-top: 1px solid #e8e8e8;
                font-size: 10px; color: #999; text-align: center;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>🕵️ AdSpy Intelligence Report</h1>
            <div class="sub">Target: {brand} &nbsp;|&nbsp; Generated: {datetime.now().strftime('%B %d, %Y %H:%M')}</div>
            <div class="sub">Model: llama-3.3-70b (Groq) &nbsp;|&nbsp; AI Engineering Bootcamp Batch 11</div>
        </div>
        <div class="content">
            {metrics_html}
            {body_html}
        </div>
        <div class="footer">
            AdSpy Intelligence Agent · AI Engineering Bootcamp Batch 11 · {datetime.now().year}
        </div>
    </body>
    </html>
    """

    HTML(string=full_html).write_pdf(pdf_file)
    print(f'✅ PDF exported: {pdf_file}')
    
    # Safe Colab download fallback
    try:
        from google.colab import files
        files.download(pdf_file)
    except ImportError:
        print("ℹ️ Note: Running in a local environment. PDF file saved to current working directory.")
        
    return pdf_file


# Export PDF
export_pdf(report, TARGET, metrics=metrics if metrics else None)

✅ PDF exported: adspy_report_1451724068428618_20260609_0943.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'adspy_report_1451724068428618_20260609_0943.pdf'